In [ ]:
from pathlib import Path
import sys

src_dir = next(p for p in [Path.cwd(), *Path.cwd().parents]
               if (p / "common.py").is_file() and (p / "feature_preference").is_dir())
sys.path.insert(0, str(src_dir))


In [ ]:
import torch
import torch.nn as nn
from tqdm.auto import tqdm
import numpy as np
import torch.nn.functional as F
import matplotlib.pyplot as plt
import seaborn as sns
import random

from common import *
import yaml
import matplotlib.colors as mcolors

import library

In [ ]:
device = torch.device("cuda")
sns.set_theme(style="whitegrid", font="cmr10", font_scale=1.9)
plt.rcParams["axes.formatter.use_mathtext"] = True
plt.rcParams.update({"pdf.fonttype": 42, "ps.fonttype": 42})
plt.rcParams.update({
    'text.usetex': False,
    'mathtext.fontset': 'cm',  # Computer Modern math font
})
colors = sns.color_palette()
colors

default plot setting

In [ ]:
with open("../linear/config.yaml", 'r', encoding='utf-8') as f:
    configs_linear = yaml.safe_load(f)
with open("../deep/config.yaml", 'r', encoding='utf-8') as f:
    configs_deep = yaml.safe_load(f)
with open("../mlp/config.yaml", 'r', encoding='utf-8') as f:
    configs_mlp = yaml.safe_load(f)

config = configs_linear

In [ ]:
def sample(batch, sizes, device, mul=0.01):
    x_out = torch.zeros((batch, 0), device=device)
    for size in sizes:
        rand = torch.rand((batch, 1), device=device).round() * 2 - 1
        x_out = torch.cat([x_out, rand.repeat(1, size)], dim=-1)
    epsilon1 = mul * torch.randn_like(x_out)
    epsilon2 = mul * torch.randn_like(x_out)
    return x_out + epsilon1, x_out + epsilon2

sample makes [+1, -1] datas

In [ ]:
def create_segment_vectors(sizes):
    total_length = sum(sizes)
    cumsum = torch.cumsum(torch.tensor(sizes), dim=-1)

    result = torch.zeros((len(sizes), total_length))

    for i in range(len(sizes)):
        start_idx = cumsum[i - 1] if i > 0 else 0
        end_idx = cumsum[i]

        result[i, start_idx:end_idx] = 1

    return result

In [ ]:
loss_types = "BarlowTwins"

batch = config["batch"]
sizes = config["sizes"]
if "sizes" in config[loss_types]:
    sizes = config[loss_types]["sizes"]
s_in = sum(sizes)
s_out = len(sizes)

In [ ]:
sample_data = sample(batch, sizes, device)
gamma = torch.bmm(sample_data[0].unsqueeze(dim=2), sample_data[1].unsqueeze(dim=1))  # b x n x n
gamma = gamma + gamma.permute(0, 2, 1)
gamma = torch.mean(gamma, dim=0).squeeze() / 2  # n x n
sns.heatmap(gamma.cpu())
plt.show()
gamma_l, gamma_v = torch.linalg.eig(gamma)
gamma_l = gamma_l.real.cpu()
gamma_v = gamma_v.real.cpu()
print(gamma_l, gamma_v)

In [ ]:
output_dict = np.load("data.npy", allow_pickle=True).item()

In [ ]:
e = create_segment_vectors(sizes).to(device)

lambdas_render_barlow = ["0", "0.1", "0.5", "1"]

In [ ]:
# print(output_dict.keys())
# epochs = config[loss_types]["epochs"]
# print(loss_types)
# if loss_types == "BarlowTwins_subspace":
#     pass
# if "BarlowTwins" in loss_types or "VICReg" in loss_types:
#     lambdas_render = lambdas_render_barlow
# else:
#     lambdas_render = [1]
# for lambd in lambdas_render:
#     if "BarlowTwins" in loss_types or "VICReg" in loss_types:
#         output_target = output_dict[loss_types][lambd]
#     else:
#         output_target = output_dict[loss_types]
#     losses = output_target["loss"]
#     eiglist = output_target["eiglist"]
#     feature_alignment = output_target["feature_alignment"]
#     s0 = output_target["s0"]
#
#     dldt = smooth_data(dt(losses), 9)
#     # plt.plot(dldt)
#
#     fig, axs = plt.subplots(1, 3, figsize=(20, 4))
#
#     axs[0].plot(losses, color=colors[2], linewidth=6, alpha=0.5)
#     critical_pos = []
#     starting = 0
#     for i in range(len(eiglist)):
#         find = np.argmin(dldt[starting:])
#         if dldt[starting + find] < -0.001:
#             critical_pos.append(starting + find)
#             starting += find + 200
#         else:
#             break
#
#     if loss_types == "BarlowTwins":
#         axs[0].set_ylim(-0.05, s_out * 1.15)
#     for pos in critical_pos:
#         axs[0].axvline(pos, color='black', linestyle='--')
#     axs[0].set_xlabel("Step")
#     axs[0].set_ylabel("Loss")
#
#     for i in range(len(eiglist)):
#         axs[1].plot(eiglist[i], linewidth=6, color=colors[i], alpha=0.5,
#                     label=(r"$\lambda_" + ("l$" if i == 0 else "s$")))
#     for pos in critical_pos:
#         axs[1].axvline(pos, color='black', linestyle='--')
#     # axs[1].set_ylim(-0.05, 1.10)
#     axs[1].set_xlabel("Step")
#     axs[1].set_ylabel("C Eigenvalue $\lambda$")
#     axs[1].legend()
#
#     if config["network"] != "mlp":
#         if loss_types == "BarlowTwins":
#             t0 = 1440
#             t = np.linspace(0, epochs, epochs)
#             y1 = gamma_l[0].item() / (
#                         gamma_l[0].item() + s0[0].item() ** (-2) * np.exp(-8 * gamma_l[0].item() * t / t0))
#             y2 = gamma_l[1].item() / (
#                         gamma_l[1].item() + s0[1].item() ** (-2) * np.exp(-8 * gamma_l[1].item() * t / t0))
#             predicted_loss = (y1 - 1) ** 2 + (y2 - 1) ** 2
#             axs[0].plot(predicted_loss, linestyle='--', color=colors[2], linewidth=2.5)
#             axs[1].plot(t, y1, linestyle='--', color=colors[0], linewidth=2.5)
#             axs[1].plot(t, y2, linestyle='--', color=colors[1], linewidth=2.5)
#             axs[2].plot(t, np.sqrt(y1), linestyle='--', color=colors[0], linewidth=2.5)
#             axs[2].plot(t, np.sqrt(y2), linestyle='--', color=colors[1], linewidth=2.5)
#
#         for i in range(len(feature_alignment)):
#             k = feature_alignment[i]
#             axs[2].plot(k, label=(r"${\text{FA}(e_l)}$" if i == 0 else r"${\text{FA}(e_s)}$"), alpha=0.5,
#                         linewidth=6, color=colors[i])
#         for pos in critical_pos:
#             axs[2].axvline(pos, color='black', linestyle='--')
#         # axs[2].set_ylim(-0.05, 1.10)
#         axs[2].legend()
#         axs[2].set_xlabel("Step")
#         axs[2].set_ylabel(r"$\|We_i\|_2$")
#     print(critical_pos)
#
#     plt.tight_layout()
#     if not ("BarlowTwins" in loss_types or "VICReg" in loss_types):
#         plt.savefig(f"figure/{loss_types.lower()}/dim_loss_eig_align_{loss_types}.pdf", format="pdf", bbox_inches='tight')
#     # plt.show()

In [ ]:
fig, axs = plt.subplots(1, 3, figsize=(26, 4.5))
epochs = config[loss_types]["epochs"]
color_continuas = sns.color_palette("rocket", n_colors=len(config[loss_types]["lambdas"]))
for index, lambd in enumerate(config[loss_types]["lambdas"]):
    output_target = output_dict[loss_types][lambd]
    losses = output_target["loss"]
    eiglist = output_target["eiglist"]
    feature_alignment = output_target["feature_alignment"]
    s0 = output_target["s0"]

    dldt = smooth_data(dt(losses), 7)
    # plt.plot(dldt)
    # plt.show()


    axs[0].plot(losses, color=color_continuas[index], linewidth=5, alpha=0.9, label=(r"$\alpha = " + f"{lambd}" + "$"))
    axs[0].legend(ncols=2)
    critical_pos = []
    starting = 0
    for i in range(len(eiglist)):
        find = np.argmin(dldt[starting:])
        if dldt[starting + find] < -0.002:
            critical_pos.append(starting + find)
            starting += find + 100
        else:
            break

    axs[0].set_ylim(-0.05, s_out * 1.15)
    # for pos in critical_pos:
    #     axs[0].axvline(pos, color='black', linestyle='--')
    axs[0].set_xlabel("Step")
    axs[0].set_ylabel("Loss")

    for i in range(len(eiglist)):
        axs[1].plot(eiglist[i], linewidth=4, color=color_continuas[index], alpha=0.9,
                    label=(r"$\lambda_" + ("l$" if i == 0 else "s$")), linestyle='--' if i == 1 else '-.')
    # for pos in critical_pos:
    #     axs[1].axvline(pos, color='black', linestyle='--')
    # axs[1].set_ylim(-0.05, 1.10)
    axs[1].set_xlabel("Step")
    axs[1].set_ylabel(r"C Eigenvalue $\lambda$")

    t0 = 1450
    t = np.linspace(0, epochs, epochs)
    y1 = gamma_l[0].item() / (
                gamma_l[0].item() + s0[0].item() ** (-2) * np.exp(-8 * gamma_l[0].item() * t / t0))
    y2 = gamma_l[1].item() / (
                gamma_l[1].item() + s0[1].item() ** (-2) * np.exp(-8 * gamma_l[1].item() * t / t0))
    predicted_loss = (y1 - 1) ** 2 + (y2 - 1) ** 2
    axs[0].plot(predicted_loss, color='blue', linewidth=2.5)
    axs[1].plot(t, y1, color='blue', linewidth=2.5)
    axs[1].plot(t, y2, color='blue', linewidth=2.5)

    for i in range(len(feature_alignment)):
        k = feature_alignment[i]
        axs[2].plot(k, label=(r"${\text{FA}(e_l)}$" if i == 0 else r"${\text{FA}(e_s)}$"), alpha=0.9,
                    linewidth=4, color=color_continuas[index], linestyle='--' if i == 1 else '-.')
    # for pos in critical_pos:
    #     axs[2].axvline(pos, color='black', linestyle='--')
    # axs[2].set_ylim(-0.05, 1.10)
    # axs[2].legend()
    axs[2].set_xlabel("Step")
    axs[2].set_ylabel(r"$\|We_i\|_2$")
    axs[2].plot(t, np.sqrt(y1), color='blue', linewidth=2.5)
    axs[2].plot(t, np.sqrt(y2), color='blue', linewidth=2.5)
    print(critical_pos)
plt.tight_layout()
plt.savefig(f"figure/{loss_types.lower()}/loss_lambda_all.pdf", format="pdf", bbox_inches='tight')
plt.show()

In [ ]:
fig, axs = plt.subplots(1, 1, figsize=(10, 5))
epochs = config[loss_types]["epochs"]
color_continuas = sns.color_palette("rocket", n_colors=len(config[loss_types]["lambdas"]))
for index, lambd in enumerate(config[loss_types]["lambdas"]):
    output_target = output_dict[loss_types][lambd]
    losses = output_target["loss"]
    eiglist = output_target["eiglist"]
    feature_alignment = output_target["feature_alignment"]
    s0 = output_target["s0"]

    dldt = smooth_data(dt(losses), 7)

    axs.plot(losses, color=color_continuas[index], linewidth=5, alpha=0.9, label=(r"$\alpha = " + f"{lambd}" + "$"))
    axs.legend(ncols=2)
    critical_pos = []
    starting = 0
    for i in range(len(eiglist)):
        find = np.argmin(dldt[starting:])
        if dldt[starting + find] < -0.002:
            critical_pos.append(starting + find)
            starting += find + 100
        else:
            break

    axs.set_xlim(-0.05, 7000)
    axs.set_ylim(-0.05, s_out * 1.15)
    axs.set_xlabel("Step")
    axs.set_ylabel("Loss")

    t0 = 1450
    t = np.linspace(0, epochs, epochs)
    y1 = gamma_l[0].item() / (
                gamma_l[0].item() + s0[0].item() ** (-2) * np.exp(-8 * gamma_l[0].item() * t / t0))
    y2 = gamma_l[1].item() / (
                gamma_l[1].item() + s0[1].item() ** (-2) * np.exp(-8 * gamma_l[1].item() * t / t0))
    predicted_loss = (y1 - 1) ** 2 + (y2 - 1) ** 2
    axs.plot(predicted_loss, color='blue', linewidth=2.5)
    print(critical_pos)
plt.tight_layout()
plt.savefig(f"figure/{loss_types.lower()}/loss_only_lambda_all.pdf", format="pdf", bbox_inches='tight')
plt.show()

In [ ]:
epochs = config[loss_types]["epochs"]
lr = float(config[loss_types]["lr"])

if "BarlowTwins" in loss_types:
    lambdas_render = lambdas_render_barlow
    color_continuas = sns.color_palette("husl", len(config[loss_types]["lambdas"]))
else:
    lambdas_render = [1]
    color_continuas = sns.color_palette("husl", 1)
for index, lambd in enumerate(lambdas_render):
    fig, axs = plt.subplots(1, 1, figsize=(14, 3.7))
    if "BarlowTwins" in loss_types:
        output_target = output_dict[loss_types][lambd]
    else:
        output_target = output_dict[loss_types]
    losses = output_target["loss"]
    eiglist = output_target["eiglist"]
    feature_alignment = output_target["feature_alignment"]
    s0 = output_target["s0"]

    for i in range(2):
        plt.plot(smooth_data(np.array(dt(eiglist[i])) / lr, window_size=10), alpha=0.5, linewidth=5, color=colors[i],
                 label=r"$d\lambda_" + ("l" if i == 0 else "s") + "/dt$")

    if loss_types == "BarlowTwins":
        dlossdt = -1 * smooth_data(np.array(dt(losses)), window_size=10)
        axs.plot(dlossdt / lr, alpha=0.7, linestyle='--', color='black', linewidth=1.5, label=r"$-d\mathcal{L}/dt$")
        axs.hlines([gamma_l[0] * 2], xmin=-1, xmax=epochs * 1.1, color=colors[0], linewidth=3, alpha=0.8, linestyle='--')
        axs.hlines([gamma_l[1] * 2], xmin=-1, xmax=epochs * 1.1, color=colors[1], linewidth=3, alpha=0.8, linestyle='--')
        axs.set_yticks([0, 10, 20, ])

        y_pos = [text.get_position()[1] for text in axs.get_yticklabels()]  #[:-1]
        y_txt = [text.get_text() for text in axs.get_yticklabels()]  #[:-1]

        new_pos0 = gamma_l[0] * 2
        new_txt0 = r"$2\gamma_l$"
        new_pos1 = gamma_l[1] * 2
        new_txt1 = r"$2\gamma_s$"

        y_pos.append(new_pos0)
        y_pos.append(new_pos1)
        y_txt.append(new_txt0)
        y_txt.append(new_txt1)
        axs.set_yticks(y_pos, y_txt)
        axs.set_ylim(-0.5, gamma_l[0] * 2 * 2)
    axs.set_xlim(0, epochs)
    axs.set_xlabel("Step")
    # axs.set_ylabel("Value")
    axs.legend(loc='upper right')
    plt.savefig(f"figure/{loss_types.lower()}/dim_dcdt_dldt_{lambd}.pdf", format="pdf", bbox_inches='tight')
    plt.show()

In [ ]:
epochs = config[loss_types]["epochs"]
lr = float(config[loss_types]["lr"])
fig, axs = plt.subplots(1, 1, figsize=(12, 4))

if "BarlowTwins" in loss_types:
    lambdas_render = lambdas_render_barlow

    for index, lambd in enumerate(lambdas_render):
        output_target = output_dict[loss_types][lambd]
        losses = output_target["loss"]
        eiglist = output_target["eiglist"]
        feature_alignment = output_target["feature_alignment"]
        s0 = output_target["s0"]
        subspace_alignment = output_target["subspace_alignment"]

        #||A^TB||^2/2
        axs.plot(subspace_alignment, alpha=0.6, color=colors[index], linewidth=5, label=r"Barlow Twins, $\lambda = " + f"{lambd}$")
else:
    lambdas_render = [1]
    color_continuas = colors
    output_target = output_dict[loss_types]
    losses = output_target["loss"]
    eiglist = output_target["eiglist"]
    feature_alignment = output_target["feature_alignment"]
    s0 = output_target["s0"]
    subspace_alignment = output_target["subspace_alignment"]

    #||A^TB||^2/2
    axs.plot(subspace_alignment, alpha=0.6, color=colors[0], linewidth=5, label=f"{loss_types}")
axs.set_xlim(0, epochs)
axs.set_ylim(-0.05, 1.05)
axs.set_xlabel("Step")
axs.set_ylabel("Subspace Alignment")
axs.legend()
plt.savefig(f"figure/{loss_types.lower()}/dim_alignment.pdf", format="pdf", bbox_inches='tight')
plt.show()

In [ ]:
lr = float(config[loss_types]["lr"])

if loss_types == "BarlowTwins":
    lambdas_render = lambdas_render_barlow
    color_continuas = sns.color_palette("husl", len(config[loss_types]["lambdas"]))
else:
    lambdas_render = [1]
    color_continuas = sns.color_palette("husl", 1)
for index, lambd in enumerate(lambdas_render):
    if loss_types == "BarlowTwins":
        output_target = output_dict[loss_types][lambd]
    else:
        output_target = output_dict[loss_types]
    losses = output_target["loss"]
    eiglist = output_target["eiglist"]
    feature_alignment = output_target["feature_alignment"]
    s0 = output_target["s0"]
    uwe = output_target["feature_weight_r_singular_alignment"]
    epochs = uwe.shape[-1]

    fig = plt.figure(figsize=(22, 10))
    ax = fig.add_subplot(projection='3d')
    for i in range(2):
        ax.plot(np.arange(0, epochs, 1), reject_outliers(np.abs(uwe[0][i]), m=0.2), reject_outliers(np.abs(uwe[1][i]), m=0.2), alpha=0.5,
                linewidth=5, color=colors[i], label=(r"$\text{Proj}_{U^{(\leq 2)}}(We_" + ("l" if i == 0 else "s") + ")$"))
    if loss_types == "BarlowTwins":
        t0 = 1450
        t = np.linspace(0, epochs, epochs)
        y1 = gamma_l[0].item() / (
                    gamma_l[0].item() + s0[0].item() ** (-2) * np.exp(-8 * gamma_l[0].item() * t / t0))
        y2 = gamma_l[1].item() / (
                    gamma_l[1].item() + s0[1].item() ** (-2) * np.exp(-8 * gamma_l[1].item() * t / t0))
        ax.plot(np.arange(0, epochs, 1), np.sqrt(y1), 0, color=colors[0], linestyle='--')
        ax.plot(np.arange(0, epochs, 1), 0, np.sqrt(y2), color=colors[1], linestyle='--')
    ax.set_xlabel("Step", labelpad=6)
    ax.set_ylabel(r"$u_1$", labelpad=6)
    ax.set_zlabel(r"$u_2$", labelpad=6)
    ax.set_box_aspect(aspect=None, zoom=0.98)
    #center right
    fig.subplots_adjust(left=0, right=1, bottom=0, top=1)
    plt.legend(loc='upper right', bbox_to_anchor=(1, 0.9))
    plt.savefig(f"figure/{loss_types.lower()}/dim_3d.pdf", format="pdf", bbox_inches='tight')
    plt.title(loss_types)
    plt.show()

#3d plot x: u1, y: u2, z: step, line1: We1, line2: We2
#use uwe[i][j], as uwe[0] means x, uwe[1] means z

